In [1]:
import jax 
from jax import lax
from jax import random as jrnd
from jax import tree_util as jtu
from jax import numpy as jnp
from jax.numpy import linalg as jnpla
from jax.scipy import linalg as jspla

from matplotlib import pyplot as plt

from time import time
from collections.abc import Callable
from typing import Any

from numerics.spectral import *
from numerics.sobol import *

jax.config.update('jax_enable_x64', True)

Point generation methods (maybe implement more, but we'll see. This'll do for now)

In [2]:
def _pts_mc(a:float|jax.Array, b:float|jax.Array, n:int, d:int, key:jax.Array = jrnd.key(0)):
    a = jnp.broadcast_to(a, (d,))
    b = jnp.broadcast_to(b, (d,))
    x = jrnd.uniform(key, (n, d), minval = a, maxval = b)
    return x

def _pts_rqmc(a:float|jax.Array, b:float|jax.Array, n:int, d:int, key:jax.Array = jrnd.key(0)):
    a = jnp.broadcast_to(a, (d,))
    b = jnp.broadcast_to(b, (d,))
    x = sobol_scrambled(n, d, key)
    x = x * (b - a) + a
    return x

_sampler_dict = {'mc':_pts_mc,
               'rqmc':_pts_rqmc}

Ugh polyharmonic splines that I thought would be a way more important part of this project than they are. I still think they might outperform VEGAS+, especially for geospatial integration. 

In [71]:
# polyharmonic spline
eps:float = 1e-14
def _phs(x: jax.Array, c: jax.Array, r: float, k: int = 2):
    """
    Computes the polyharmonic spline value r^k * log(r).
    
    Parameters
    ----------
    x : jax.Array
        Point array.
    c : jax.Array
        Center array.
    r : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    jax.Array
        Spline value.
    """
    # Easy
    r_safe = lax.select(jnp.abs(r) < eps, 1., r)
    val = jnp.where(k % 2 == 0, jnp.log(r_safe), 1.)
    return r ** k * val

# Gradient
def _grad_phs(x: jax.Array, c: jax.Array, r: float, k: int = 2):
    """
    Computes the gradient of the polyharmonic spline.
    
    Parameters
    ----------
    x : jax.Array
        Point array.
    c : jax.Array
        Center array.
    r : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    jax.Array
        Gradient vector.
    """
    # Easy
    dxc = x - c
    r_safe = lax.select(r < eps, 1., r)
    a = jnp.where(k % 2 == 0, k * jnp.log(r_safe) + 1, k)

    return dxc * r ** (k - 2) * a

# Gradient magnitude
def _maggrad_phs(x: jax.Array, c: jax.Array, r: float, k: int = 2):
    """
    Computes the magnitude of the gradient of the polyharmonic spline.
    
    Parameters
    ----------
    x : jax.Array
        Point array.
    c : jax.Array
        Center array.
    r : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    jax.Array
        Magnitude of the gradient.
    """
    # Easy
    r_safe = lax.select(r < eps, 1., r)
    a = jnp.where(k % 2 == 0, k * jnp.log(r_safe) + 1, k)

    return jnp.abs(r ** (k - 1) * a)

# Hessian
def _hess_phs(x: jax.Array, c: jax.Array, r: float, k: int = 2):
    """
    Computes the Hessian of the polyharmonic spline.
    
    Parameters
    ----------
    x : jax.Array
        Point array.
    c : jax.Array
        Center array.
    r : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    jax.Array
        Hessian matrix.
    """
    # Precursors
    n = x.shape[0]
    dxc = x - c
    r_safe = lax.select(r < eps, 1., r)

    # Calc
    I = jnp.eye(n)
    D = jnp.outer(dxc, dxc)
    a = jnp.where(k % 2 == 0, k * jnp.log(r_safe) + 1, k)
    b = jnp.where(k % 2 == 0, (k - 1) * a + 2, (k - 1) * k)
    return r ** (k - 2) * (I * a + D * (b - a) / r ** 2)

# Gradient of magnitude of gradient (weird but helps us cluster points)
def _gmg_phs(x: jax.Array, c: jax.Array, r: float, k: int = 2):
    """
    Computes the gradient of the magnitude of the gradient of the polyharmonic spline.
    
    Parameters
    ----------
    x : jax.Array
        Point array.
    c : jax.Array
        Center array.
    r : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    jax.Array
        Gradient of the magnitude of the gradient.
    """
    # Precursors
    dxc = x - c
    r_safe = lax.select(r < eps, 1., r)
    klnr = k * jnp.log(r_safe)
    km1 = k - 1
    rkm2 = r ** (k - 2)
    
    # Calc
    a_even, a_odd = klnr + 1, k
    a = jnp.where(k % 2 == 0, a_even, a_odd)
    b_even = a * km1 + k
    b_odd = k * km1
    b = rkm2 * jnp.where(k % 2 == 0, b_even, b_odd)
    return b * (dxc / r_safe)

In [125]:
a,b = -1, 1
n = 1000
ni = 150
x = jnp.linspace(a, b, n)
xi = jnp.linspace(a, b, ni)
c = 0
r = jnp.abs(x - c)
ri = jnp.abs(xi - c)

ksel = [2, 3, 4]
csel = [0, 0.25]
clen = len(csel)

adjfn = lambda arr: jnp.cumsum(arr)

In [ ]:

def _init_state():

def _body_fn():

def vegas_rbf(f:Callable[[jax.Array], Any], g:Callable[[Any], float], a:float|jax.Array, b:float|jax.Array,
          ni:int, n_iters:int, d:int, k_poly:int = 3, 
          sampler:str = 'mc', basis_poly:str = 'M', 
          key:jax.Array = jrnd.key(0)):
      #### SETUP ####
      # Grab sampler & adaptive sampling engine
      _sampler = _sampler_dict[sampler]
      # Take proposal samples...
      x = _sampler(0, 1, ni * n_iters, d, key)
      # And slice
      xn = x.reshape(n_iters, ni, d)

      # Grab initial state...
      init_state = _init_state()
      # Scan over all samples...
      (xf, wf, yf), (x0f, w0f, y0f, y0f_raw) = lax.scan(_body_fn, init_state, xn)
      # And return quadrature points + weights + values
      return xf, yf, wf

In [6]:
# NOTE NEEDS (n - n0) % m_iters = 0
engine_dict = {'mc':_pts_mc,
               'rqmc':_pts_rqmc}

def _sched_const(i, m_iters, eps):
    return eps

def _sched_linear(i, m_iters, eps):
    return eps + (i / m_iters)

schedule_dict = {'constant':_sched_const,
                 'linear':_sched_linear}

def vrbfs(f:Callable, a:float|jax.Array, b:float|jax.Array, 
          n0:int, n_iter:int, m_iters:int, n_est:int, d:int, 
          k_phs:int = 4, k_poly:int = 3, basis_poly:str = 'M',
          engine:str = 'qmc', schedule:str = 'linear', 
          schedule_eps:float = 1., key:jax.Array = jrnd.key(0)):
    
    # SETUP
    # -----
    # Grab engine and scheduler
    _engine = engine_dict[engine]
    _scheduler = schedule_dict[schedule]
    # Split keys
    keys = jrnd.split(key, 2)
    # Broadcast bounds
    a, b = jnp.broadcast_to(a, (d,)), jnp.broadcast_to(b, (d,))
    # Calculate total number of points
    n = n0 + m_iters * n_iter
    mn = m_iters * n_iter
    # Grab unscaled x
    x_raw = _engine(a, b, n, d, keys[0])
    # (Scaling)
    def scale(x_raw):
        return 2 * (x_raw - a) / (b - a) - 1
    def unscale(x):
        return (1 + x) * (b - a) / 2 + a
    x = scale(x_raw)

    # POLYNOMIALS
    # -----------
    # Slice x to construct and update block matrix
    x0, xn = x[:n0], x[n0:].reshape(m_iters, n_iter, d)
    # Grab polynomial basis
    zeros_poly = jnp.zeros((d,))
    p0 = mv_psi(x0, basis_poly, k_poly, zeros_poly)
    pd = k_poly ** d
    p = jnp.pad(p0, ((0, n - n0 + pd), (0, 0)))

    # FORMING THE BLOCK MATRIX (Phi)
    # ------------------------------
    # Calculate distance matrix for initial points
    R0 = jnpla.norm(x0[:, None, :] - x0[None, :, :], axis = -1)
    # Calculate polyharmonic spline matrix
    #   (look up "double vmap" if confused)
    phs_map = jax.vmap(jax.vmap(_phs, in_axes = (None, 0, 0, None)), in_axes = (0, None, 0, None))
    phs0 = phs_map(x0, x0, R0, k_phs)
    # Grab zeros for the block matrix.
    # Form the full block matrix
    zeros_01 = jnp.zeros((n0, mn))
    zeros_12 = jnp.zeros((mn, pd))
    eye = jnp.eye(mn)
    zeros_22 = jnp.zeros((pd, pd))
    Phi0 = jnp.block([[phs0, zeros_01, p0],
                      [zeros_01.T, eye, zeros_12],
                      [p0.T, zeros_12.T, zeros_22]])
    # Form the inverse 
    #### TODO: IMPLEMENT THIS INVERSE
    Phi_inv0 = jnpla.inv(Phi0)

    # CALCULATING WEIGHTS
    # -------------------
    # Evaluate the function at these initial points
    y = jax.vmap(f)(unscale(x0))
    # Pad with zeros for unevaluated points and 
    #   polynomials
    y = jnp.pad(y, (0, n - n0 + pd))
    w0 = Phi_inv0 @ y

    # Make gmg map
    gmg_phs_map = jax.vmap(jax.vmap(_gmg_phs, in_axes = (None, 0, 0, None)), in_axes = (0, None, 0, None))

    # Update for a single iteration
    def update(carry, xi):
        i,x,p,y,Phi_inv,w = carry
        # EVALUATE GRADIENT OF MAGNITUDE OF GRADIENT
        #   FOR EXISTING INTERPOLANT
        # ---------------------------
        # Grab start/end indices
        j_start = n0 + i * n_iter
        j_end = j_start + n_iter
        # Get distance matrix between all points
        #   And points for iteration (ones not yet added
        #   will have zero weights so they'll get masked.
        #   This step lets us keep shape homogeneity)
        Ri = jnpla.norm(x[:, None, :] - xi[None, :, :], axis = -1)
        # Grab gmg of interpolant using analytic formula...
        gmgi = gmg_phs_map(x, xi, Ri, k_phs)
        # ...and weights
        gmgi = jnp.einsum('ijk,i->jk', gmgi, w[:-pd])
        # We also normalize the gradient update
        gmgi = gmgi / jnpla.vector_norm(gmgi, axis = 0).sum() * _scheduler(i, m_iters, schedule_eps)

        # UPDATING INTERNALS
        # -----------------
        # Update x (the "where" term is to wrap massive updates back around)
        xi = xi + gmgi
        xi = jnp.where(xi > 1., (xi % 2) - 1, xi)
        xi = jnp.where(xi < -1, (xi % 2) - 1, xi)
        x = lax.dynamic_update_slice_in_dim(x, xi, j_start, axis = 0)
        # Update polynomials 
        pi = mv_psi(xi, basis_poly, k_poly, zeros_poly)
        p = lax.dynamic_update_slice_in_dim(p, pi, j_start, axis = 0)
        # Update y
        yi = jax.vmap(f)(unscale(xi))
        y = lax.dynamic_update_slice(y, yi, (j_start,))
        
        #  CONDITIONING UPDATE FOR PHI
        # -----------------------
        # Build a little PHS matrix for these updated points
        Ri2 = jnpla.norm(x[:, None, :] - xi[None, :, :], axis = -1)
        phsi = phs_map(x, xi, Ri2, k_phs)
        # And attach polynomial evaluations
        Phii = jnp.concat([phsi, pi.T], axis = 0)
        # Grab indices for masking
        ii,jj = jnp.indices(Phii.shape)
        # Set diagonal equal to -0.5 so that
        #   it will cancel the identity when we update
        #   Phi0
        diag_mask = ii - jj == j_start
        Phii = Phii - 0.5 * diag_mask
        # Mask out unnecessary points
        unused_mask = jnp.logical_or(ii < j_end, ii > n - pd)
        Phii = jnp.where(unused_mask, Phii, 0.)

        # UPDATING PHI, w
        # --------------------------------
        #   Our matrix now admits the decomposition 
        #   Phi_stari ⊗ diag_mask + diag_mask ⊗ Phi_stari. 
        #   This is the same as B @ C @ B.T, where
        #   B is a block matrix of our vectors and flipped indices
        #   and C is a reversed identity. 
        #   (Note that C is selv-inverse, so we just call it C_inv!)
        B = jnp.concat([Phii, diag_mask[:, ::-1]], axis = -1)
        C_inv = jnp.flip(jnp.eye(2 * n_iter), axis = 0)
        # Perform the Woodbury Update
        schur = jnpla.inv(C_inv + B.T @ Phi_inv @ B)
        Phi_inv = Phi_inv - Phi_inv @ B @ schur @ B.T @ Phi_inv
        # Update w
        w = Phi_inv @ y
        carry = (i + 1,x,p,y,Phi_inv,w)

        return carry, unscale(x)
    
    # Scanning
    init = (0,x,p,y,Phi_inv0,w0)
    test = update(init, xn[0])
    final,xc = lax.scan(update, init, xn)
    _,xf,pf,yf,Phi_invf,wf = final

    # Evaluate the integrand using the interpolant
    x_est = engine_dict[engine](-1., 1., n_est, d)
    p_est = mv_psi(x_est, basis_poly, k_poly, zeros_poly)
    R_est = jnpla.norm(x[:, None, :] - x_est[None, :, :], axis = -1)
    phs_est = phs_map(x, x_est, R_est, k_phs)
    Phi_est = jnp.concat([phs_est, p_est.T], axis = 0)
    y_est = wf @ Phi_est
    return y_est.mean()

In [7]:
from scipy.integrate import nquad

def f(x):
    x = jnp.array(x)
    term1 = (4 - 2.1*x[0]**2 + x[0]**4/3) * x[0]**2
    term2 = jnp.prod(x)
    term3 = 4 * x[1]**2 * (x[1]**2 - 1)
    return term1 + term2 + term3

def test_methods(f, a, b, n0, n_iter, m_iters, d):
    n = n0 + n_iter * m_iters
    k_phs = 4

    a,b = jnp.broadcast_to(a, (d,)), jnp.broadcast_to(b, (d,))
    ab = [(ai,bi) for (ai,bi) in zip(a,b)]

    f_scp = lambda *x: f(x)
    I = nquad(f_scp, ab)[0]

    # Monte Carlo estimate
    x_mc = _pts_mc(a, b, n, 2, jrnd.key(1))
    y_mc = jax.vmap(f)(x_mc)
    w_mc = jnp.full_like(y_mc, jnp.prod(b - a) / n)
    I_mc = y_mc @ w_mc


    print('True: %s, MC: %s'%(I, I_mc))
    # VRBFS estimate
    for i in range(6):
        I_vrbfs= vrbfs(f, a, b, n0, n_iter, m_iters, n, 2, k_phs, engine = 'qmc', schedule = 'linear', key = jrnd.key(i))
        print(I_vrbfs)

test_methods(f, -jnp.pi, jnp.pi, 20, 25, 4, 2)


True: 3268.6508153907625, MC: 3643.8260483971694


KeyError: 'qmc'